THU THẬP DỮ LIỆU

In [ ]:
import requests
from bs4 import BeautifulSoup
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import dask.dataframe as dd
import os
from datetime import date, timedelta
import plotly.express as px

page = requests.get("https://www.worldometers.info/coronavirus")

# Soup is a variable containing the HTML of the webpage
soup = BeautifulSoup(page.content, 'lxml')

# Search for the table and extracting it
table = soup.find('table', attrs={'id': 'main_table_countries_yesterday'})
rows = table.find_all("tr", attrs={"style": ""})

data = []
for i, item in enumerate(rows):
    if i == 0:
        data.append(item.text.strip().split("\n")[:16])
    else:
        data.append(item.text.strip().split("\n")[:15])

data[0][13] = data[0][13] + data[0][14]
data[0][14] = 'Population'
data[1].pop()
data[1].insert(0, ' ')

dt = pd.DataFrame(data[1:], columns=data[0][:15]) # Formatting the header
df = dd.from_pandas(dt, npartitions=1)

yesterday = (date.today() - timedelta(days = 1)).strftime(r"%d-%m-%Y")
print(yesterday)
path = "Extracted_data"
if not os.path.exists(path):
    os.makedirs(path)
#df.compute().to_csv(os.path.join(path,"data_covid19_{}.csv".format(yesterday)))

04-05-2022


SỐ DÒNG LẶP

In [ ]:
dt.duplicated().sum()

0

TIỀN XỬ LÍ DỮ LIỆU

In [ ]:
df_list = []
startDay = 25
endDay = 31   #date 1/5
lastest_date = "2022-05-01"
dataFilePath = "Extracted_data/data_covid19_{}-2022.csv"
#load and insert date column to each dataframe
for i in range(startDay,endDay+1):
  if i == 31:
    date_df = pd.read_csv(dataFilePath.format("01-05"),index_col = [0])
    date_df.insert(2,'Date',pd.to_datetime("01-05-2022", dayfirst = True))
  #not 1/5
  else:
    date_df = pd.read_csv(dataFilePath.format(str(i)+'-04'),index_col = [0])
    date_df.insert(2,'Date',pd.to_datetime("{}-2022".format(str(i)+'-04')))
  df_list.append(date_df)
#folder to store pre-processed data
if not os.path.exists("Pre-processed_data"):
    os.makedirs("Pre-processed_data")
#data preprocessing including dropping unnecessary rows, filling in na values and converting string to numeric data
for df in df_list:
  df.drop(df[df['#'] == 'Total:'].index, inplace = True)
  df.fillna('0', inplace = True)
  for column in list(df)[3:]:
    df[column] = df[column].apply(lambda x : str(x).strip('+').replace(',','') if not str(x).isspace() else '0')
    df[column] = pd.to_numeric(df[column], errors = 'coerce', downcast = 'unsigned')
  for i in range(13,16):
    df.iloc[0,i] = df.iloc[:,i].sum()
  #save pre-processed data
  df.to_csv("Pre-processed_data" + "/data_covid19_{}.csv".format(df['Date'][0]))
general_df = pd.concat(df_list, axis = 0, ignore_index = True)
general_df

,#,"Country,Other",Date,TotalCases,NewCases,TotalDeaths,NewDeaths,TotalRecovered,NewRecovered,ActiveCases,"Serious,Critical",Tot Cases/1M pop,Deaths/1M pop,TotalTests,Tests/1M pop,Population
0,,World,2022-04-25,510080177,403941,6245802,1952,463138404,1049158,40695971,42448,65439,801.3,6257874437,404555957,7884959285
1,1,China,2022-04-25,203334,2680,4776,51,169380,2982,29178,274,141,3.0,160000000,111163,1439323776
2,2,USA,2022-04-25,82747175,38858,1018718,186,80506860,41509,1221597,1415,247359,3045.0,1001729381,2994507,334522343
3,3,India,2022-04-25,43062097,2011,522223,0,42523311,1970,16563,698,30658,372.0,834717702,594272,1404606308
4,4,Brazil,2022-04-25,30355919,6456,662777,76,29411813,27459,281329,8318,140994,3078.0,63776166,296221,215299307
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1542,219,Falkland Islands,2022-05-01,128,0,0,0,0,0,0,0,34925,0.0,8632,2355252,3665
1543,222,Marshall Islands,2022-05-01,17,0,0,0,14,0,3,0,284,0.0,0,0,59920
1544,225,Niue,2022-05-01,9,0,0,0,7,0,2,0,5468,0.0,0,0,1646
1545,226,Micronesia,2022-05-01,7,0,0,0,1,0,6,0,60,0.0,0,0,117226


KIỂU DỮ LIỆU CÁC CỘT TRONG BẢNG

In [ ]:
general_df.dtypes

#                           object
Country,Other               object
Date                datetime64[ns]
TotalCases                  uint32
NewCases                    uint32
TotalDeaths                 uint32
NewDeaths                   uint16
TotalRecovered              uint32
NewRecovered                uint32
ActiveCases                  int64
Serious,Critical            uint16
Tot Cases/1M pop            uint32
Deaths/1M pop              float64
TotalTests                   int64
Tests/1M pop                 int64
Population                   int64
dtype: object

In [ ]:
latest_df = general_df[(general_df['Country,Other'] != 'World') & (general_df['Date'] == lastest_date)]
totalDeaths_df = latest_df.sort_values(by = 'TotalDeaths', ascending = False).head(50)
fig = px.bar(totalDeaths_df, x = 'TotalDeaths', y = 'Country,Other',
             orientation = 'h', height = 1000, title = 'Top 50 nước có tổng số tử vong cao nhất')
fig.update_layout(yaxis = {'categoryorder' : 'total ascending'})
fig.show()

In [ ]:
fig = px.scatter(latest_df, x = 'TotalCases', y = 'TotalDeaths', 
                 height = 1000, title = 'Total cases and total deaths relationship', trendline = 'ols')
fig.show()

In [ ]:
fig = px.scatter(latest_df, x = 'Serious,Critical', y = 'NewDeaths', 
                 height = 1000, title = 'Serious Critical and New deaths relationship', trendline = 'ols')
fig.show()

In [ ]:
fig = px.scatter(latest_df, x = 'TotalCases', y = 'TotalRecovered', 
                 height = 1000, title = 'Total cases and Total recovered relationship', trendline = 'ols')
fig.show()

In [ ]:
countries = ['Vietnam']
someCountries_df = general_df[(general_df['Country,Other'].isin(countries)) 
                  & (general_df['Country,Other'] != 'World')]
fig = px.line(someCountries_df, x = 'Date', y = 'NewCases', title = 'Số ca nhiễm mới ở Việt Nam qua từng ngày', markers = True)
fig.show()

In [ ]:
death_percentage = []
for i in range(len(latest_df)):
  totalCases = latest_df.iloc[i,latest_df.columns.get_loc('TotalCases')]
  totalDeaths = latest_df.iloc[i,latest_df.columns.get_loc('TotalDeaths')]
  death_percentage.append((totalDeaths/totalCases) * 100)
fig = px.bar(latest_df.head(50), x = 'Country,Other', y = death_percentage[:50])
fig.update_layout(xaxis_title = 'Countries', yaxis_title = '% of Total deaths/Total cases', xaxis = {'categoryorder' : 'category ascending'})
fig.show()

In [ ]:
active_in_pop_percentage = []
for i in range(len(latest_df)):
  actCases = latest_df.iloc[i,latest_df.columns.get_loc('ActiveCases')]
  pop = latest_df.iloc[i,latest_df.columns.get_loc('Population')]
  active_in_pop_percentage.append((actCases/pop) * 100)
fig = px.bar(latest_df, x = 'Country,Other', y = active_in_pop_percentage)
fig.update_layout(xaxis_title = 'Countries', yaxis_title = '% of Active cases/Population', xaxis = {'categoryorder' : 'category ascending'})
fig.show()

In [ ]:
totalCasesSorted_df = general_df[general_df['Date'] == lastest_date].sort_values(by = 'TotalCases', ascending = False)
totalCasesSorted_df.drop(totalCasesSorted_df[totalCasesSorted_df['Country,Other'] == 'World'].index, inplace = True)
totalCasesSorted_df = totalCasesSorted_df.reset_index(drop = True)
totalCasesSorted_df.loc[totalCasesSorted_df.index > 20, 'Country,Other'] = 'Others'
#totalCasesSorted_df
fig = px.pie(totalCasesSorted_df, names = 'Country,Other', values = 'TotalCases', title = 'Phần trăm số ca nhiễm của các nước so với toàn thế giới')
fig.show()